# Non-Linear Activation Functions: A Short Comparison

*Generated from the research paper by an LLM. Review the code before relying on its results.*

# Non-Linear Activation Functions: A Short Comparison

This notebook implements the short teaching paper *"Non-Linear Activation Functions: A Short Comparison"*. The paper describes itself as a teaching document, not a published paper.

**Problem.** A stack of linear layers is still one linear map, however deep it is. A network built only from linear layers cannot learn a curved decision boundary.

**Key idea.** Apply a non-linear function $f$ element-wise after each layer, $h = f(Wx+b)$. The choice of $f$ matters for more than expressiveness. Through the chain rule, $f'$ decides how well gradients flow backwards:
- **Sigmoid** has $\sigma'(x)\le 1/4$, so gradients shrink exponentially with depth (*vanishing gradients*).
- **ReLU-family** functions have derivative 1 for active units, so gradients survive.

**What we will build, from scratch in NumPy:**
1. A numerical check that two linear layers collapse into one (Eq. 1).
2. Six activations and their derivatives (Eqs. 2–7), verified with finite differences.
3. A measurement of vanishing gradients versus depth (Eq. 8).
4. The paper's experiment: a 2–16–16–1 MLP trained on two-moons with full-batch gradient descent, once per activation, starting from identical initial weights. We use hand-written backprop and BCE loss (Eq. 9).

Everything runs on CPU in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 100

## 1. Why non-linearity is needed

A layer computes $h = Wx + b$. Two stacked linear layers give

$$W_2(W_1x+b_1)+b_2 = \underbrace{W_2W_1}_{W}\,x + \underbrace{W_2b_1+b_2}_{b},$$

which is a single linear layer. We check this numerically. Then we insert a ReLU between the layers and show that no such single $(W, b)$ reproduces the result any more.

In [ ]:
rng = np.random.default_rng(0)
W1, b1 = rng.normal(size=(5, 3)), rng.normal(size=5)
W2, b2 = rng.normal(size=(4, 5)), rng.normal(size=4)
x = rng.normal(size=3)

two_layers = W2 @ (W1 @ x + b1) + b2
W, b = W2 @ W1, W2 @ b1 + b2            # the collapsed single layer from Eq. (1)
one_layer = W @ x + b
print('two linear layers :', two_layers)
print('collapsed layer   :', one_layer)
print('max abs difference:', np.abs(two_layers - one_layer).max())

# With a ReLU in between, the collapsed (W, b) no longer reproduces the output
with_relu = W2 @ np.maximum(0, W1 @ x + b1) + b2
print('\nwith ReLU in between:', with_relu)
print('difference vs collapsed linear map:', np.abs(with_relu - one_layer).max())

## 2. Activation functions and their derivatives

| Name | $f(x)$ | $f'(x)$ |
|---|---|---|
| Sigmoid (2) | $\sigma(x)=\frac{1}{1+e^{-x}}$ | $\sigma(x)(1-\sigma(x))$ |
| Tanh (3) | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $1-\tanh^2(x)$ |
| ReLU (4) | $\max(0,x)$ | $1$ if $x>0$, else $0$ |
| Leaky ReLU (5), $\alpha=0.01$ | $x$ if $x>0$, else $\alpha x$ | $1$ if $x>0$, else $\alpha$ |
| GELU (6) | $x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\left(\sqrt{2/\pi}\,(x+0.044715x^3)\right)\right]$ | see below |

The paper gives the tanh approximation of GELU but not its derivative. Write $u=\sqrt{2/\pi}(x+0.044715x^3)$ and $t=\tanh u$. The product and chain rules then give

$$\text{GELU}'(x) \approx \tfrac12(1+t) + \tfrac12 x(1-t^2)\sqrt{2/\pi}\,(1+3\cdot0.044715\,x^2).$$

We also add an **identity** ("no activation") function. It is not in the paper's table, but it serves as a baseline that shows Section 1 in practice.

The sigmoid is implemented in a numerically stable way, so that $e^{-x}$ never overflows.

In [ ]:
ALPHA = 0.01
C_GELU = np.sqrt(2.0 / np.pi)

def sigmoid(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out

def d_sigmoid(x):
    s = sigmoid(x); return s * (1 - s)

def tanh(x):   return np.tanh(x)
def d_tanh(x): return 1 - np.tanh(x) ** 2

def relu(x):   return np.maximum(0.0, x)
def d_relu(x): return (np.asarray(x) > 0).astype(float)

def leaky_relu(x):   return np.where(x > 0, x, ALPHA * x)
def d_leaky_relu(x): return np.where(np.asarray(x) > 0, 1.0, ALPHA)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(C_GELU * (x + 0.044715 * x ** 3)))
def d_gelu(x):
    t = np.tanh(C_GELU * (x + 0.044715 * x ** 3))
    return 0.5 * (1 + t) + 0.5 * x * (1 - t ** 2) * C_GELU * (1 + 3 * 0.044715 * x ** 2)

def identity(x):   return np.asarray(x, dtype=float)
def d_identity(x): return np.ones_like(np.asarray(x, dtype=float))

ACTS = {
    'sigmoid':    (sigmoid, d_sigmoid),
    'tanh':       (tanh, d_tanh),
    'relu':       (relu, d_relu),
    'leaky_relu': (leaky_relu, d_leaky_relu),
    'gelu':       (gelu, d_gelu),
    'identity':   (identity, d_identity),   # baseline, not in the paper
}

### Verifying the derivatives

Backprop is only as correct as $f'$, so we compare each analytic derivative with a central finite difference, $\frac{f(x+\epsilon)-f(x-\epsilon)}{2\epsilon}$. The grid deliberately avoids $x=0$ exactly, because ReLU's kink there makes the finite difference meaningless. We also check how close the tanh approximation of GELU is to the exact $x\Phi(x)$, with $\Phi(x)=\frac12[1+\text{erf}(x/\sqrt2)]$.

In [ ]:
xs = np.linspace(-4, 4, 400)   # even count, so 0 is not on the grid
eps = 1e-5
for name, (f, df) in ACTS.items():
    fd = (f(xs + eps) - f(xs - eps)) / (2 * eps)
    print(f'{name:11s} max |analytic - finite diff| = {np.abs(df(xs) - fd).max():.2e}')

gelu_exact = xs * 0.5 * (1 + erf(xs / np.sqrt(2)))
print(f'\nGELU tanh-approximation max error vs exact x*Phi(x): {np.abs(gelu(xs) - gelu_exact).max():.2e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs = np.linspace(-4, 4, 400)
for name, (f, df) in ACTS.items():
    if name == 'identity':
        continue
    axes[0].plot(xs, f(xs), label=name)
    axes[1].plot(xs, df(xs), label=name)
axes[0].set_title('Activation $f(x)$'); axes[0].set_ylim(-1.5, 4)
axes[1].set_title("Derivative $f'(x)$")
axes[1].axhline(0.25, color='k', ls=':', lw=1)
axes[1].text(-3.9, 0.28, r"$\sigma'(x)\leq 1/4$", fontsize=9)
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5); ax.legend()
plt.tight_layout(); plt.show()

### Softmax (output layer only)

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$$

We subtract $\max_j z_j$ before exponentiating. This leaves the result unchanged and prevents overflow. For two classes, softmax with logits $[z, 0]$ gives exactly $\sigma(z)$. That is why the paper's binary experiment can use a single sigmoid output unit.

In [ ]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

z = np.array([2.0, 1.0, 0.1, 1000.0])
print('softmax of large logits (stable):', softmax(z), ' sum =', softmax(z).sum())
zz = np.linspace(-5, 5, 7)
print('softmax([z,0])[0]:', softmax(np.stack([zz, np.zeros_like(zz)], axis=1))[:, 0])
print('sigmoid(z)       :', sigmoid(zz))

## 3. Vanishing gradients

In an $L$-layer network the gradient reaching layer 1 is a product of one Jacobian per layer, and each Jacobian contains $\text{diag}(f'(z_l))\,W_l$. Because $\sigma'(x)\le\frac14$,

$$\left\|\frac{\partial\mathcal L}{\partial h_1}\right\| \le \left\|\frac{\partial\mathcal L}{\partial h_L}\right\| \prod_{l=2}^L\|W_l\| \cdot\left(\tfrac14\right)^{L-1}.$$

**Experiment.** We build deep networks of width 32 and depth $L=1,\dots,20$, initialised with $W\sim\mathcal N(0, 1/\text{fan-in})$. We push a batch of random inputs forward, inject a random gradient at the top, and backprop it down. We then report

$$\text{ratio}(L) = \frac{\|\partial\mathcal L/\partial W_1\|}{\|\partial\mathcal L/\partial W_L\|},$$

which measures how much of the signal survives the trip to the first layer. The results are averaged over 10 random networks.

In [ ]:
def first_layer_grad_ratio(act, depth, width=32, batch=256, seed=0):
    f, df = ACTS[act]
    rng = np.random.default_rng(seed)
    Ws = [rng.normal(0, 1 / np.sqrt(width), (width, width)) for _ in range(depth)]
    a = rng.normal(size=(batch, width))
    acts, zs = [a], []
    for W in Ws:                       # forward pass
        z = a @ W; zs.append(z); a = f(z); acts.append(a)
    g = rng.normal(size=a.shape)       # dL/dh_L: an arbitrary upstream gradient
    grad_norms = [None] * depth
    for l in reversed(range(depth)):   # backward pass
        gz = g * df(zs[l])
        grad_norms[l] = np.linalg.norm(acts[l].T @ gz)
        g = gz @ Ws[l].T
    return grad_norms[0] / grad_norms[-1]

depths = np.arange(1, 21)
plt.figure(figsize=(7, 4.5))
for act in ['sigmoid', 'tanh', 'relu', 'leaky_relu', 'gelu']:
    ratios = [np.mean([first_layer_grad_ratio(act, L, seed=s) for s in range(10)]) for L in depths]
    plt.semilogy(depths, ratios, 'o-', ms=3, label=act)
plt.semilogy(depths, 0.25 ** (depths - 1), 'k:', label=r'$(1/4)^{L-1}$ reference')
plt.xlabel('depth L'); plt.ylabel(r'$\|\nabla W_1\| / \|\nabla W_L\|$')
plt.title('Gradient reaching the first layer vs depth'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the plot.** The sigmoid curve falls roughly geometrically with depth, as Eq. (8) predicts: each layer multiplies the gradient by about $\|W_l\|\cdot\sigma'(z)\lesssim\frac14\|W_l\|$.

Tanh decays more slowly because $\tanh'(0)=1$. ReLU-family curves stay much flatter. For ReLU with this $1/\text{fan-in}$ initialisation, roughly half the units are active, so there is still some drift. He initialisation, $\mathcal N(0, 2/\text{fan-in})$, is designed to cancel that drift.

The exact slopes depend on the initialisation scale. Try changing it in `first_layer_grad_ratio`.

## 4. The experiment: 2–16–16–1 MLP on two-moons

This section follows the paper's setup:
- **Data:** two-moons, 1,000 points, noise 0.2, 80/20 train/test split.
- **Model:** $2\to16\to16\to1$. Both hidden layers use the activation under test, and the output uses a sigmoid.
- **Loss:** binary cross-entropy,
$$\mathcal L = -\frac1N\sum_{n=1}^N\left[y_n\log\hat y_n + (1-y_n)\log(1-\hat y_n)\right].$$
- **Optimiser:** full-batch gradient descent with $\eta=0.1$ for 2,000 steps. Every activation starts from the **same** initial weights.
- **Recorded:** the training-loss curve, test accuracy, and $\|\partial\mathcal L/\partial W_1\|$ at step 0.

The paper does not specify the initialisation, so we use $W\sim\mathcal N(0, 1/\text{fan-in})$ and $b=0$.

**Backprop.** With a sigmoid output and BCE loss, the gradient with respect to the output pre-activation simplifies to $\frac{\partial\mathcal L}{\partial z_{\text{out}}} = \frac{\hat y - y}{N}$. For each layer $l$, working backwards:

$$\frac{\partial\mathcal L}{\partial W_l} = a_{l-1}^\top \delta_l,\qquad \frac{\partial\mathcal L}{\partial b_l}=\textstyle\sum_n \delta_{l,n},\qquad \delta_{l-1} = (\delta_l W_l^\top)\odot f'(z_{l-1}).$$

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
print('train:', Xtr.shape, ' test:', Xte.shape)

plt.figure(figsize=(5, 3.5))
plt.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap='coolwarm', s=8)
plt.title('Two-moons training data'); plt.show()

In [ ]:
def init_params(sizes=(2, 16, 16, 1), seed=0):
    rng = np.random.default_rng(seed)
    return [[rng.normal(0, 1 / np.sqrt(fi), (fi, fo)), np.zeros(fo)]
            for fi, fo in zip(sizes[:-1], sizes[1:])]

def copy_params(params):
    return [[W.copy(), b.copy()] for W, b in params]

def forward(params, X, act):
    f, _ = ACTS[act]
    a, acts, zs = X, [X], []
    for i, (W, b) in enumerate(params):
        z = a @ W + b
        zs.append(z)
        a = sigmoid(z) if i == len(params) - 1 else f(z)   # sigmoid output layer
        acts.append(a)
    return a.ravel(), (acts, zs)

def bce(yhat, y, eps=1e-12):
    yhat = np.clip(yhat, eps, 1 - eps)
    return -np.mean(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))

def backward(params, cache, y, act):
    _, df = ACTS[act]
    acts, zs = cache
    delta = (acts[-1] - y[:, None]) / len(y)       # dL/dz_out for sigmoid + BCE
    grads = [None] * len(params)
    for i in reversed(range(len(params))):
        W, _ = params[i]
        grads[i] = (acts[i].T @ delta, delta.sum(axis=0))
        if i > 0:
            delta = (delta @ W.T) * df(zs[i - 1])
    return grads

### Gradient check of the full network

Before training, we confirm that `backward` matches finite differences of the loss for every activation. We perturb each weight of the first layer, which is the layer whose gradient the paper reports.

In [ ]:
def grad_check(act, eps=1e-6):
    params = init_params(seed=1)
    Xs, ys = Xtr[:50], ytr[:50]
    _, cache = forward(params, Xs, act)
    gW1 = backward(params, cache, ys, act)[0][0]
    num = np.zeros_like(gW1)
    for idx in np.ndindex(gW1.shape):
        old = params[0][0][idx]
        params[0][0][idx] = old + eps; lp = bce(forward(params, Xs, act)[0], ys)
        params[0][0][idx] = old - eps; lm = bce(forward(params, Xs, act)[0], ys)
        params[0][0][idx] = old
        num[idx] = (lp - lm) / (2 * eps)
    return np.abs(num - gW1).max() / (np.abs(num).max() + 1e-12)

for act in ACTS:
    print(f'{act:11s} relative error of dL/dW1: {grad_check(act):.2e}')

In [ ]:
def train(act, params0, lr=0.1, steps=2000):
    params = copy_params(params0)          # identical starting weights for every activation
    losses, g0 = [], None
    for t in range(steps):
        yhat, cache = forward(params, Xtr, act)
        losses.append(bce(yhat, ytr))
        grads = backward(params, cache, ytr, act)
        if t == 0:
            g0 = np.linalg.norm(grads[0][0])   # first-layer gradient norm at start
        for p, (gW, gb) in zip(params, grads):
            p[0] -= lr * gW
            p[1] -= lr * gb
    test_acc = np.mean((forward(params, Xte, act)[0] > 0.5) == yte)
    return dict(params=params, losses=np.array(losses), test_acc=test_acc, grad0=g0)

params0 = init_params(seed=0)
results = {act: train(act, params0) for act in ACTS}

print(f"{'activation':11s} {'final loss':>10s} {'test acc':>9s} {'||dL/dW1|| at t=0':>19s}")
for act, r in results.items():
    print(f"{act:11s} {r['losses'][-1]:10.4f} {r['test_acc']:9.3f} {r['grad0']:19.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for act, r in results.items():
    axes[0].plot(r['losses'], label=act, ls='--' if act == 'identity' else '-')
axes[0].set_xlabel('GD step'); axes[0].set_ylabel('training BCE'); axes[0].set_title('Training loss')
axes[0].legend()

names = list(results)
axes[1].bar(names, [results[a]['test_acc'] for a in names], color='tab:green')
axes[1].set_ylim(0.5, 1.0); axes[1].set_title('Test accuracy')
axes[2].bar(names, [results[a]['grad0'] for a in names], color='tab:purple')
axes[2].set_yscale('log'); axes[2].set_title(r'$\|\partial\mathcal{L}/\partial W_1\|$ at step 0')
for ax in axes[1:]:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

### Decision boundaries

The identity network can only draw a straight line, as Section 1 predicts. The non-linear networks can bend the boundary around the moons.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2, 3, 250), np.linspace(-1.5, 2, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (act, r) in zip(axes.ravel(), results.items()):
    p = forward(r['params'], grid, act)[0].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=20, cmap='coolwarm', alpha=0.6)
    ax.contour(xx, yy, p, levels=[0.5], colors='k')
    ax.scatter(Xte[:, 0], Xte[:, 1], c=yte, cmap='coolwarm', s=8, edgecolors='k', linewidths=0.2)
    ax.set_title(f"{act}  (test acc {r['test_acc']:.3f})")
plt.tight_layout(); plt.show()

### Robustness across seeds

The paper notes that *exact numbers depend on the random seed*. We repeat the whole experiment for 5 initialisation seeds and report the mean ± standard deviation. As before, all activations share the same initial weights within each seed.

In [ ]:
seeds = range(5)
summary = {act: {'acc': [], 'loss': [], 'g0': []} for act in ACTS}
for s in seeds:
    p0 = init_params(seed=s)
    for act in ACTS:
        r = train(act, p0)
        summary[act]['acc'].append(r['test_acc'])
        summary[act]['loss'].append(r['losses'][-1])
        summary[act]['g0'].append(r['grad0'])

print(f"{'activation':11s} {'test acc':>16s} {'final loss':>18s} {'grad0 (mean)':>13s}")
for act, d in summary.items():
    print(f"{act:11s} {np.mean(d['acc']):.3f} ± {np.std(d['acc']):.3f}   "
          f"{np.mean(d['loss']):.4f} ± {np.std(d['loss']):.4f}   {np.mean(d['g0']):.2e}")

### Dead ReLU units

Section 3 warns that a ReLU unit whose pre-activation is $\le 0$ for **every** input gets zero gradient forever. Leaky ReLU avoids this through its slope $\alpha$.

We count such units in the trained networks. On this small, well-initialised problem the count is often zero. To see the failure mode clearly, we also train with a deliberately large learning rate: big updates can push units into the dead region.

In [ ]:
def count_dead(params, act):
    _, (acts, zs) = forward(params, Xtr, act)
    return [int(np.sum(np.all(z <= 0, axis=0))) for z in zs[:-1]]  # per hidden layer

for lr in [0.1, 2.0]:
    for act in ['relu', 'leaky_relu']:
        dead = [sum(count_dead(train(act, init_params(seed=s), lr=lr, steps=500)['params'], act)) for s in range(5)]
        print(f'lr={lr:<4} {act:11s} dead units (of 32) per seed: {dead}')
print('\nNote: for leaky ReLU, a "dead" unit is negative everywhere but still gets gradient alpha*... so it can recover.')

## How our results relate to the paper's Table 1

The paper reports only *qualitative* expectations, and our numbers come from one small configuration. Compare them against the table below rather than treating them as a reproduction of definitive results.

| Activation | Paper's expectation | What to look for above |
|---|---|---|
| Sigmoid | slowest; smallest first-layer gradient | highest loss curve; smallest bar in the gradient-norm plot |
| Tanh | faster than sigmoid | loss drops faster than sigmoid |
| ReLU / Leaky ReLU / GELU | fastest; similar final accuracy | lowest loss curves, near-identical accuracies |

Two extra observations go beyond the paper:
- The identity baseline plateaus at the accuracy of a linear classifier, around 85–88% on this data. This is Eq. (1) in action.
- With only two hidden layers, vanishing gradients slow sigmoid down but do not stop it. The deep-network plot in Section 3 shows the effect far more dramatically. Given enough steps, sigmoid would usually catch up on this shallow network. Exact numbers vary with seed, initialisation scheme and step count.

## Summary

- **Without non-linearity, depth is useless.** Two linear layers collapse into one, which we verified numerically. The identity-activation MLP can only draw a straight decision boundary.
- **The derivative decides trainability.** $\sigma'\le\frac14$ makes gradients decay roughly as $(\frac14)^{L-1}$ with depth. Tanh is better because $\tanh'(0)=1$. ReLU, Leaky ReLU and GELU pass gradients through active units nearly unchanged.
- **On two-moons**, the ReLU-family networks reach low loss fastest and with similar accuracy. Sigmoid starts with the smallest first-layer gradient and learns slowest, matching the paper's qualitative Table 1.
- **Dead units** are a real risk for ReLU under aggressive learning rates. Leaky ReLU's small negative slope lets them recover.
- **Softmax** is the multi-class generalisation of the sigmoid output and is used only at the output layer.

## Ideas for extending the experiment
1. **Go deeper.** Train 2–16–…–16–1 networks with 5–10 hidden layers on two-moons and watch sigmoid fail completely while ReLU still trains.
2. **Initialisation.** Compare $\mathcal N(0,1/\text{fan-in})$ (LeCun), Xavier/Glorot, and He initialisation for each activation, reusing `first_layer_grad_ratio` and `train`.
3. **Optimisers.** Replace plain GD with momentum or Adam. Adaptive step sizes partly hide vanishing gradients.
4. **More activations.** Add ELU, SiLU/Swish ($x\sigma(x)$) or softplus to `ACTS`; you only need $f$ and $f'$.
5. **Multi-class.** Use `make_blobs` or 3-class spirals with a softmax output and cross-entropy loss. The output-layer gradient is again $\hat y - y$ (one-hot).
6. **Exact GELU.** Use $x\Phi(x)$ with the derivative $\Phi(x)+x\phi(x)$ and compare it with the tanh approximation.